In [1]:
import pandas as pd

weights = pd.read_csv(
    "data/call_weights_by_zip.csv",
    dtype={"zipcode": "string"},
)

print(f"{len(weights)} ZIPs")
display(weights.head())

37 ZIPs


,zipcode,calls
0,11207,12473
1,11212,12176
2,11201,11460
3,11208,9873
4,11226,8791


In [2]:
modzcta = pd.read_csv("data/modzcta.csv", dtype="string")
modzcta.columns = modzcta.columns.str.strip().str.lower()

In [3]:
display(modzcta[["modzcta", "label"]].head(10))

,modzcta,label
0,10001,"10001, 10118"
1,10002,10002
2,10003,10003
3,10026,10026
4,10004,10004
5,10005,10005
6,10006,10006
7,10007,10007
8,10009,10009
9,10010,10010


In [4]:
lookup = modzcta[["modzcta", "label"]].copy()
lookup["zipcode"] = lookup["label"].str.split(",")
lookup = lookup.explode("zipcode")
lookup["zipcode"] = lookup["zipcode"].str.strip()

lookup = (
    lookup.loc[lookup["zipcode"].str.fullmatch(r"\d{5}", na=False),
               ["zipcode", "modzcta"]]
    .dropna(subset=["modzcta"])
    .drop_duplicates()
)

joined = weights.merge(lookup, on="zipcode", how="left", validate="one_to_one")
unmatched = joined.loc[joined["modzcta"].isna()]

print(f"Matched {len(joined) - len(unmatched)} of {len(joined)} ZIPs")
display(unmatched[["zipcode", "calls_train", "calls_test"]])
display(unmatched[["calls_train", "calls_test"]].sum().rename("unmatched_calls"))

Matched 37 of 37 ZIPs


KeyError: "['calls_train', 'calls_test'] not in index"

In [6]:
unmatched = joined.loc[joined["modzcta"].isna(), ["zipcode", "calls"]]
assert unmatched.empty, f"ZIPs still missing a MODZCTA:\n{unmatched}"

area_weights = (
    joined.groupby("modzcta", as_index=False)["calls"]
    .sum()
)

assert area_weights["calls"].sum() == weights["calls"].sum()

print(f"{len(area_weights)} demand areas")
display(area_weights.sort_values("calls", ascending=False).head(10))

37 demand areas


,modzcta,calls
5,11207,12473
10,11212,12176
0,11201,11460
6,11208,9873
24,11226,8791
4,11206,8664
30,11233,8495
1,11203,7205
22,11224,6650
33,11236,6522


In [8]:
matched_zips = joined.dropna(subset=["modzcta"])
zips_per_area = matched_zips.groupby("modzcta")["zipcode"].transform("nunique")

shared = matched_zips.loc[
    zips_per_area.gt(1),
    ["modzcta", "zipcode", "calls"],
].sort_values(["modzcta", "zipcode"])

display(shared)
display(area_weights.loc[area_weights["modzcta"].isin(shared["modzcta"])])

,modzcta,zipcode,calls


,modzcta,calls


In [10]:
import geopandas as gpd

boundaries = gpd.read_file("data/modzcta.geojson")
boundaries.columns = boundaries.columns.str.strip().str.lower()
boundaries["modzcta"] = boundaries["modzcta"].astype("string").str.strip()

demand_areas = boundaries[["modzcta", "geometry"]].merge(
    area_weights, on="modzcta", how="inner", validate="one_to_one"
)

missing_areas = area_weights.loc[
    ~area_weights["modzcta"].isin(demand_areas["modzcta"]), "modzcta"
]
assert missing_areas.empty, f"Missing polygons for {missing_areas.tolist()}"
assert demand_areas.geometry.notna().all()
assert not demand_areas.geometry.is_empty.any()

# Pick an interior point for each polygon and convert to lat/lon.
points = demand_areas.to_crs("EPSG:2263").representative_point().to_crs("EPSG:4326")

demand = (
    demand_areas.assign(lat=points.y, lon=points.x)
    .rename(columns={"modzcta": "area_id"})
    [["area_id", "lat", "lon", "calls"]]
)

print(f"{len(demand)} demand areas")
display(demand.head())
display(demand.loc[demand["area_id"].eq("11211")])

37 demand areas


,area_id,lat,lon,calls
0,11201,40.694644,-73.990312,11460
1,11203,40.648697,-73.933898,7205
2,11204,40.621012,-73.985932,2877
3,11205,40.696517,-73.966369,3461
4,11206,40.702414,-73.941854,8664


,area_id,lat,lon,calls
9,11211,40.71147,-73.957366,5023


In [11]:
demand.shape

(37, 4)

In [12]:
demand.to_csv("data/demand.csv", index=False)

In [14]:
# Compute centroids in a local projected CRS for NYC.
areas_projected = demand_areas.to_crs("EPSG:2263")
centroids = areas_projected.geometry.centroid

# Check each centroid against its own polygon, then convert to lat/lon.
inside_area = areas_projected.geometry.covers(centroids)
centroids_wgs84 = centroids.to_crs("EPSG:4326")

demand_centroids = (
    demand_areas.assign(lat=centroids_wgs84.y, lon=centroids_wgs84.x)
    .rename(columns={"modzcta": "area_id"})
    [["area_id", "lat", "lon", "calls"]]
)

display(demand_centroids.head())
print(f"Centroids outside their area: {(~inside_area).sum()}")
display(demand_centroids.loc[~inside_area, ["area_id", "lat", "lon", "calls"]])

,area_id,lat,lon,calls
0,11201,40.694608,-73.989581,11460
1,11203,40.649574,-73.934362,7205
2,11204,40.618777,-73.984824,2877
3,11205,40.694628,-73.966205,3461
4,11206,40.701960,-73.942352,8664


Centroids outside their area: 0


,area_id,lat,lon,calls


In [15]:
demand_centroids.shape

(37, 4)

In [16]:
demand_centroids.to_csv("data/demand_centroids.csv", index=False)